# 00 — vLLM duman testi (smoke test)

**Amacı tek cümlede:** LLM katmanının *gerçekten* çalıştığının ilk kanıtını üretmek.

Bu notebook'a kadar `src/extraction/llm/**` altındaki kod **hiç çalıştırılmamıştı**:
sunucu ayakta değilken hata sessizce yutuluyordu ve sonuç "LLM hiçbir alan bulamadı"
olarak görünüyordu. Burada üç şey kanıtlanır:

1. Sunucunun hangi **yapılandırılmış çıktı (structured output)** modunu kabul ettiği
   — tahmin edilerek değil, 1 token'lık gerçek isteklerle **ölçülerek** bulunur.
2. Gerçek bir Türkçe katılım bankacılığı kampanya metninden **şema-geçerli JSON**
   üretildiği.
3. Alan bazlı güven skorunun **logprob'lardan** hesaplandığı (modelin kendi
   beyanından değil).

## Mimari kural (pazarlık dışı)

Colab burada **yalnızca bir koşucudur**. Teslim edilen sistemin Colab'a
**çalışma zamanı bağımlılığı yoktur**: ngrok/cloudflared/tünel **kullanılmaz**.
vLLM aynı makinede `localhost` üzerinde kalkar; yerel kurulumdan tek farkı
`VLLM_URL` ortam değişkenidir (şartname §5.9, CLAUDE.md §2).


In [ ]:
# GPU kontrolü — A100 bekleniyor. Colab'da: Runtime > Change runtime type > A100.
!nvidia-smi


In [ ]:
# vLLM kurulumu (Colab). ~3-5 dk sürer.
# Sürüm SABİTLENMEZ: hangi yapılandırılmış-çıktı parametresinin geçerli olduğu
# sürüme göre değişir ve kod bunu ÖLÇEREK bulur (clients.py yetenek pazarlığı).
# Sabit sürüm, kodun taşınabilirlik iddiasını test etmeden geçirirdi.
!pip install -q vllm
!python -c "import vllm; print('vllm', vllm.__version__)"


In [ ]:
# Depoyu klonla. (Zaten klonluysa bu hücreyi atla ve REPO yolunu elle ayarla.)
import os, pathlib, subprocess

REPO_URL = os.environ.get("ANATOLIA_REPO_URL", "")  # ör. https://github.com/<kullanici>/<repo>.git
REPO = pathlib.Path("/content/anatoliaaI/app")

if not REPO.exists():
    assert REPO_URL, "REPO_URL'i doldurun ya da depoyu elle yükleyin."
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/anatoliaaI"],
                   check=True)

os.chdir(REPO)
print("cwd:", os.getcwd())


In [ ]:
# ---------------------------------------------------------------------------
# Model seçimi — lisans kısıtı MUTLAKTIR (şartname §5.10, CLAUDE.md §7/§20).
# Yalnız Apache-2.0 / MIT ağırlıklar. Qwen3 ailesi Apache-2.0'dır.
# Trendyol-LLM-8B-T1 BLOKELİ: taban model zinciri doğrulanana dek kullanılmaz
# (bkz. docs/model-license-audit.md).
#
# Kalite tavanı (A100 40GB'de deney için):
#   "Qwen/Qwen3-32B"        -> en yüksek kalite, AWQ/kısa bağlam gerekebilir
#   "Qwen/Qwen3-14B-AWQ"    -> 4-bit, A100'e rahat sığar, hızlı
#   "Qwen/Qwen3-8B"         -> güvenli varsayılan
# Teslim/demo (CPU, GPU yok): Qwen3-4B GGUF + Ollama (bkz. 'CPU demo' bölümü).
#
# NOT: depo adlarını çalıştırmadan önce huggingface.co üzerinde doğrulayın;
# nicemlenmiş (AWQ) varyantların adları zamanla değişebilir.
# ---------------------------------------------------------------------------
MODEL = "Qwen/Qwen3-8B"
PORT = 8001                 # 8000 API'nin kendi portu; çakışmasın diye 8001
MAX_LEN = 8192              # 6 few-shot örneği + uzun kampanya metni sığsın


In [ ]:
# vLLM'i AYNI MAKİNEDE, localhost'ta bir alt süreç (subprocess) olarak kaldır.
#
# ÖNEMLİ — mimari kural: burada ngrok/cloudflared/tünel YOKTUR ve olmayacaktır.
# Şartname §5.9 sistemin dış servislere bağımlı olmadan çalışmasını istiyor.
# Colab yalnızca bir KOŞUCUdur (runner); teslim edilen sistem Colab'a bağlanmaz.
# Yerel makine ile Colab arasındaki tek fark bir ORTAM DEĞİŞKENİDİR (VLLM_URL).
import subprocess, sys, time, urllib.request, os

log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL,
     "--port", str(PORT),
     "--max-model-len", str(MAX_LEN),
     "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT)

def wait_ready(port, timeout_s=1200):
    """Model yüklenene kadar bekle. İlk indirme dakikalar sürebilir."""
    started = time.time()
    while time.time() - started < timeout_s:
        if server.poll() is not None:
            raise RuntimeError("vLLM süreci öldü — /content/vllm.log dosyasına bakın")
        try:
            with urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2):
                return time.time() - started
        except Exception:
            time.sleep(5)
    raise TimeoutError("vLLM zamanında ayağa kalkmadı")

print(f"hazır ({wait_ready(PORT):.0f} sn)")
os.environ["VLLM_URL"] = f"http://localhost:{PORT}"
os.environ["VLLM_MODEL"] = MODEL
os.environ["LLM_BACKEND"] = "vllm"


## 1) Yetenek pazarlığı — hangi mod çalışıyor?

`guided_json` vLLM'in ESKİ parametre adıdır. Ad en az iki kez değişti ve hangisinin
geçerli olduğu sürüme göre farklı. Kod sürüm **tahmin etmez**: dört modu sırayla
1 token'lık gerçek bir istekle dener, ilk çalışanı hafızaya alır.

Aşağıdaki çıktı rapora ve sunuma girer — "hangi kanıtla konuşuyoruz" sorusunun cevabıdır.


In [ ]:
from src.extraction.llm.clients import VLLMClient
from src.extraction.llm.schema import guided_json_schema

client = VLLMClient()          # VLLM_URL / VLLM_MODEL env'den okunur
schema = guided_json_schema()

mode = client.negotiate(schema)
print("SEÇİLEN MOD:", mode)
print("pazarlık günlüğü:")
for m, sonuc in client.negotiation_log:
    print(f"  {m:22s} -> {sonuc}")


## 2) Gerçek kampanya metninde çıkarım

Metin uydurulmaz: önce depodaki altın (gold) örneklerden, yoksa toplanmış ham
veriden okunur. Hiçbiri yoksa hücre açıkça uyarır.


In [ ]:
import json, pathlib

gold_path = pathlib.Path("data/gold/gold.sample.json")
texts = []
if gold_path.exists():
    texts = [item["text"] for item in json.loads(gold_path.read_text(encoding="utf-8"))]

# Daha uzun/gerçekçi bir örnek varsa ham veriden de al.
for p in sorted(pathlib.Path("data/processed").glob("*.txt"))[:2]:
    texts.append(p.read_text(encoding="utf-8")[:2000])

assert texts, "Çalıştırılacak metin bulunamadı (data/gold veya data/processed boş)."
for i, t in enumerate(texts):
    print(f"[{i}] {t[:120]}...")


In [ ]:
from src.extraction.llm.extractor import LLMExtractor

ex = LLMExtractor(client)      # strict değil: ham çıktıyı da görmek istiyoruz
sonuc = ex.call(texts[0])      # 12 alanın tamamı sorulur

print("hata      :", sonuc.error)
print("gecikme   :", f"{sonuc.latency_ms:.0f} ms")
print("mod       :", sonuc.structured_mode)
print("onarım    :", sonuc.retries)
print("logprob # :", len(sonuc.logprobs))
print("\n--- HAM ÇIKTI ---")
print(sonuc.raw_text)


## 3) Şema geçerliliği — iddia edilen şey ölçülür

"JSON döndü" yetmez; **istenen şemaya uyan** JSON döndüğü doğrulanmalı.
`jsonschema` kuruluysa tam doğrulama yapılır; değilse en azından zorunlu
anahtarların tamamının üretildiği kontrol edilir (model bir alanı ATLAYAMAZ,
bulamadığını açıkça `null` yazmak zorundadır).


In [ ]:
from src.extraction.llm.parse import parse_llm_json
from src.extraction.llm.schema import EXTRACTION_FIELDS

obj, err = parse_llm_json(sonuc.raw_text)
assert obj is not None, f"ayrıştırılamadı: {err}"

eksik = [f for f in EXTRACTION_FIELDS if f not in obj]
fazla = [k for k in obj if k not in EXTRACTION_FIELDS]
print("eksik anahtar:", eksik or "yok")
print("şema dışı anahtar:", fazla or "yok")

try:
    import jsonschema
    jsonschema.validate(obj, guided_json_schema())
    print("jsonschema doğrulaması: GEÇTİ")
except ImportError:
    print("jsonschema kurulu değil -> `pip install jsonschema` ile tam doğrulama yapılabilir")


## 4) Alanlar + güven kaynağı

`confidence_source` her satırda yazar:

- `logprob` — skor modelin token seçimindeki tereddüdünden **ölçüldü**,
- `self_reported` — model kendi söyledi (kalibre değil; Ollama yolunda tek seçenek).

Bu ayrım kalibrasyon (ECE) hesabında iki kaynağın karışmaması için gereklidir.


In [ ]:
for f in sonuc.fields:
    dogru_span = f.verify_span(texts[0])
    print(f"{f.field_name:22s} {str(f.canonical_value)[:34]:36s} "
          f"conf={f.confidence:.3f} ({f.confidence_source})  span_ok={dogru_span}")

print("\nçağrı istatistikleri:", ex.summary())


## 5) Tüm örneklerde toplu koşu

Kalan gold metinlerinde de koşarak hata oranını ölçelim. `LLMCallResult` sayesinde
"kaç çağrı yapıldı / kaçı ayrıştırılamadı / kaçı onarımla kurtarıldı" sorularının
sayısal cevabı var — eskiden hepsi tek bir boş listeye çöküyordu.


In [ ]:
ex.reset_stats()
for t in texts:
    r = ex.call(t)
    print(f"{len(r.fields):2d} alan | {r.latency_ms:7.0f} ms | onarım={r.retries} | "
          f"hata={r.error}")
print("\nTOPLAM:", ex.summary())


## 6) Kural + LLM hibrit (uzlaştırma)

`reconcile()` kural katmanını birincil tutar, boşlukları LLM ile doldurur.
`verify_low_conf` deney anahtarıdır: `>0` verilirse güveni eşiğin altındaki
**kural** alanları da LLM'e sorulur ve o alanlarda katman önceliği gevşer.
Varsayılan `0.0`'dır — kanıtlanmadan varsayılan yapılmaz.


In [ ]:
from src.extraction.reconcile import reconcile

for etiket, esik in [("hibrit (varsayılan)", 0.0), ("hibrit + doğrulama", 0.75)]:
    alanlar = reconcile(texts[0], llm=ex, verify_low_conf=esik)
    print(f"\n{etiket}: {len(alanlar)} alan")
    for f in sorted(alanlar, key=lambda x: x.field_name):
        print(f"  {f.field_name:22s} {str(f.canonical_value)[:30]:32s} "
              f"[{f.extractor.value}] conf={f.confidence:.2f}")


## 7) CPU demo yolu (Ollama) — teslim ortamının provası

Teslim makinesinde GPU yok. Aynı kod, aynı şema, farklı istemci:
`OLLAMA_MODEL` olarak **Qwen3-4B GGUF** (Apache-2.0) çekilir. Kodda değişen tek
şey `LLM_BACKEND` ortam değişkenidir — mimarinin taşınabilirlik iddiası budur.

Bu hücre Colab'da **gerekli değildir**; teslim makinesinde koşulacak komutları
belgelemek için buradadır.


In [ ]:
# Teslim (CPU) makinesinde:
#   curl -fsSL https://ollama.com/install.sh | sh
#   ollama pull qwen3:4b
#   export LLM_BACKEND=ollama OLLAMA_MODEL=qwen3:4b OLLAMA_NUM_CTX=8192 OLLAMA_KEEP_ALIVE=30m
#   python -m eval.ablation --gold data/gold/gold.sample.json
#
# Kod tarafında değişiklik YOK:
#   from src.extraction.llm.extractor import default_extractor
#   ex = default_extractor()     # LLM_BACKEND'e bakar
print("CPU demo adımları yukarıdaki yorumda.")


In [ ]:
# Sunucuyu kapat (GPU belleğini bırak).
server.terminate()
server.wait(timeout=60)
print("vLLM kapandı")
